# Centralised NF-ToN-IoT: XGBoost & Random Forest

**Research question:** in the federated FedGATSage results (see `paper.md` / `findings.md` in the repo), several attack classes were heavily confused with each other — `ddos`, `injection`, `password`, `scanning`, `xss` all look nearly identical in raw NetFlow fields, and federated partitioning makes it worse (each client only sees a slice of the traffic). This notebook checks whether a **centralised** model, with full access to the data and engineered flow-diversity features, can separate those classes.

This notebook clones the repo, checks out the `centralised` branch, and runs the actual scripts in `centralised/` (`preprocess.py`, `train_xgboost.py`, `train_random_forest.py`) — no duplicated code here.

Before running: attach the NF-ToN-IoT dataset as a Kaggle input (search Kaggle Datasets for "NF-ToN-IoT", or upload `datasets/nftoniot/NF-ToN-IoT.csv` as a private dataset — it isn't committed to git).

In [ ]:
REPO_URL = "https://github.com/asfi50/Fed_GNN"
BRANCH = "centralised"

!rm -rf /kaggle/working/Fed_GNN
!git clone --branch {BRANCH} --single-branch {REPO_URL} /kaggle/working/Fed_GNN
%cd /kaggle/working/Fed_GNN/centralised

In [ ]:
!pip install -q -r requirements.txt

## 1. Locate the dataset

`datasets/` is gitignored, so it doesn't come with the clone — find the CSV under `/kaggle/input/` (from the attached dataset) instead.

In [ ]:
import glob

candidates = glob.glob("/kaggle/input/**/*NF-ToN-IoT*.csv", recursive=True)
candidates = [c for c in candidates if "easy" not in c.lower() and "common" not in c.lower()]
assert candidates, "Could not find NF-ToN-IoT.csv under /kaggle/input — attach the dataset first."
RAW_PATH = candidates[0]
print("Using dataset:", RAW_PATH)

## 2. Comet ML

Uses the same API key/project convention as `experiments/fedgatsage_experiment.py` in the repo (`centralised/comet_utils.py` reads `COMET_API_KEY` from the environment, falling back to that default). Override it here if you want to log to your own workspace.

In [ ]:
import os
# os.environ["COMET_API_KEY"] = "your-key-here"  # optional override

## 3. Preprocess: engineer features on the full dataset, then balance

`--samples-per-class 15000` balances every class to 15k rows; classes with fewer than `--min-raw-count` raw rows (mitm: 1295, ransomware: 142) are dropped automatically, leaving the 8 classes from Table 2 of the paper.

In [ ]:
!python preprocess.py \
    --input "{RAW_PATH}" \
    --output data/nfton_balanced.csv \
    --samples-per-class 15000 \
    --min-raw-count 15000

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

balanced_preview = pd.read_csv("data/nfton_balanced.csv")
fig, ax = plt.subplots(figsize=(8, 4))
balanced_preview["Attack"].value_counts().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Balanced class distribution")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()
del balanced_preview

## 4. Train XGBoost

Set `--device cuda` if the notebook has a GPU accelerator attached; otherwise leave `cpu`.

In [ ]:
!python train_xgboost.py \
    --data data/nfton_balanced.csv \
    --out-dir results \
    --device cpu

## 5. Train Random Forest

In [ ]:
!python train_random_forest.py \
    --data data/nfton_balanced.csv \
    --out-dir results

## 6. Did centralised training recover the classes federated learning missed?

Recall numbers for FedGATSage (federated, NF-ToN-IoT) are hardcoded from Table 2 of `paper.md`. Compare against the recall these centralised models achieve on the same classes (reading back the classification reports the scripts just wrote to `results/`).

In [ ]:
import json

with open("results/xgboost_classification_report.json") as f:
    xgb_report = json.load(f)
with open("results/random_forest_classification_report.json") as f:
    rf_report = json.load(f)

# FedGATSage federated recall on NF-ToN-IoT, Table 2 in paper.md
fedgatsage_recall = {
    "Benign": 0.9944, "backdoor": 0.9942, "ddos": 0.5322, "dos": 1.0000,
    "injection": 0.3412, "password": 0.6226, "scanning": 0.8029, "xss": 0.9990,
}

rows = []
for cls, fed_recall in fedgatsage_recall.items():
    if cls not in xgb_report:
        continue
    rows.append({
        "class": cls,
        "fedgatsage_recall": fed_recall,
        "xgboost_recall": xgb_report[cls]["recall"],
        "random_forest_recall": rf_report[cls]["recall"],
    })

comparison = pd.DataFrame(rows).set_index("class")
comparison["xgboost_vs_fed"] = comparison["xgboost_recall"] - comparison["fedgatsage_recall"]
comparison["rf_vs_fed"] = comparison["random_forest_recall"] - comparison["fedgatsage_recall"]
comparison.to_csv("results/federated_vs_centralised_recall.csv")
comparison